Random Forest

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# =========================
# 1️⃣ Load Dataset
# =========================
df = pd.read_csv("../../data/audio_features_updated.csv")

# =========================
# 2️⃣ Clean Dataset
# =========================
df = df.select_dtypes(include=["int64", "float64"])
df = df.dropna()

X = df.drop("label", axis=1).values
y = df["label"].values

print("Dataset shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =========================
# 3️⃣ Train-Test Split
# =========================
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 4️⃣ Train-Validation Split (for tuning)
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=42
)

# =========================
# 5️⃣ SMOTE (ONLY on training set)
# =========================
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", np.bincount(y_train))

# =========================
# 6️⃣ Feature Scaling
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# =========================
# 7️⃣ Manual Hyperparameter Tuning (NO CV)
# =========================
param_grid = {    
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
}

best_score = 0
best_params = None
best_model = None

for n in param_grid["n_estimators"]:
    for d in param_grid["max_depth"]:
        for s in param_grid["min_samples_split"]:

            model = RandomForestClassifier(
                n_estimators=n,
                max_depth=d,
                min_samples_split=s,
                class_weight="balanced",
                random_state=42
            )

            model.fit(X_train, y_train)

            y_val_prob = model.predict_proba(X_val)[:, 1]
            score = roc_auc_score(y_val, y_val_prob)

            if score > best_score:
                best_score = score
                best_params = (n, d, s)
                best_model = model

print("\nBest Params:", best_params)
print("Best Validation ROC-AUC:", best_score)

# =========================
# 8️⃣ Final Evaluation on Test Set
# =========================
THRESHOLD = 0.40

y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > THRESHOLD).astype(int)

print("\nPredicted distribution:", np.bincount(y_pred))
print("Actual distribution:", np.bincount(y_test))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

roc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", roc)

Dataset shape: (141, 132)
Class distribution: [99 42]
After SMOTE: [63 63]

Best Params: (200, None, 2)
Best Validation ROC-AUC: 0.5044642857142857

Predicted distribution: [17 12]
Actual distribution: [20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.55      0.59        20
           1       0.25      0.33      0.29         9

    accuracy                           0.48        29
   macro avg       0.45      0.44      0.44        29
weighted avg       0.52      0.48      0.50        29

ROC-AUC Score: 0.5111111111111111


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# =========================
# 1️⃣ Load Dataset
# =========================
df = pd.read_csv("../../data/audio_features_full.csv")

# =========================
# 2️⃣ Clean Dataset
# =========================
df = df.select_dtypes(include=["int64", "float64"])
df = df.dropna()

X = df.drop("label", axis=1)
y = df["label"]

print("Dataset shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =========================
# 3️⃣ Train-Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 4️⃣ Pipeline (SMOTE + Scaling + Model)
# =========================
pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42))
])

# =========================
# 5️⃣ Hyperparameter Grid
# =========================
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

# =========================
# 6️⃣ GridSearch with 5-Fold CV
# =========================
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# =========================
# 7️⃣ Best Model
# =========================
best_model = grid.best_estimator_

# =========================
# 8️⃣ Predictions
# =========================
THRESHOLD = 0.40

y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > THRESHOLD).astype(int)

print("\nPredicted distribution:", np.bincount(y_pred))
print("Actual distribution:", np.bincount(y_test))

# =========================
# 9️⃣ Evaluation
# =========================
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

roc = roc_auc_score(y_test, y_prob)
print("Test ROC-AUC Score:", roc)

Dataset shape: (141, 50)
Class distribution: [99 42]
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 400}
Best CV Score: 0.6705952380952381

Predicted distribution: [17 12]
Actual distribution: [20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.55      0.59        20
           1       0.25      0.33      0.29         9

    accuracy                           0.48        29
   macro avg       0.45      0.44      0.44        29
weighted avg       0.52      0.48      0.50        29

Test ROC-AUC Score: 0.5111111111111111


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# =========================
# 1️⃣ Load Dataset
# =========================
df = pd.read_csv("../../data/audio_features_updated.csv")

# =========================
# 2️⃣ Clean Dataset
# =========================
df = df.select_dtypes(include=["int64", "float64"])
df = df.dropna()

X = df.drop("label", axis=1)
y = df["label"]

print("Dataset shape:", X.shape)
print("Class distribution:", np.bincount(y))

# =========================
# 3️⃣ Train-Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 4️⃣ Pipeline (SMOTE + Scaling + XGBoost)
# =========================
pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(
        eval_metric="logloss",
        use_label_encoder=False,
        random_state=42
    ))
])

# =========================
# 5️⃣ Hyperparameter Grid
# =========================
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 6, 10],
    "model__learning_rate": [0.01, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

# =========================
# 6️⃣ GridSearch with 5-Fold CV
# =========================
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

# =========================
# 7️⃣ Best Model
# =========================
best_model = grid.best_estimator_

# =========================
# 8️⃣ Predictions
# =========================
THRESHOLD = 0.40

y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > THRESHOLD).astype(int)

print("\nPredicted distribution:", np.bincount(y_pred))
print("Actual distribution:", np.bincount(y_test))

# =========================
# 9️⃣ Evaluation
# =========================
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

roc = roc_auc_score(y_test, y_prob)
print("Test ROC-AUC Score:", roc)

Dataset shape: (141, 132)
Class distribution: [99 42]
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Parameters: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 6, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best CV ROC-AUC: 0.6636309523809524

Predicted distribution: [24  5]
Actual distribution: [20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.85      0.77        20
           1       0.40      0.22      0.29         9

    accuracy                           0.66        29
   macro avg       0.55      0.54      0.53        29
weighted avg       0.61      0.66      0.62        29

Test ROC-AUC Score: 0.47777777777777775


c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:39:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
